# Exploratory Data Analysis (EDA) - Visualization Guide

This notebook provides a comprehensive guide on when to use different visualizations during exploratory data analysis.

## Table of Contents
1. [Univariate Analysis](#univariate-analysis)
2. [Bivariate Analysis](#bivariate-analysis)
3. [Multivariate Analysis](#multivariate-analysis)
4. [Distribution Analysis](#distribution-analysis)
5. [Correlation Analysis](#correlation-analysis)
6. [Categorical Data Analysis](#categorical-data-analysis)
7. [Time Series Analysis](#time-series-analysis)
8. [Outlier Detection](#outlier-detection)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

# Create sample dataset
np.random.seed(42)
df = pd.DataFrame({
    'Age': np.random.normal(35, 10, 200),
    'Salary': np.random.normal(60000, 15000, 200),
    'Experience': np.random.normal(10, 5, 200),
    'Department': np.random.choice(['HR', 'IT', 'Finance', 'Sales'], 200),
    'Performance': np.random.randint(1, 6, 200),
    'Date': pd.date_range('2020-01-01', periods=200, freq='D')
})

print("Sample Dataset:")
print(df.head())
print(f"\nShape: {df.shape}")

## 1. Univariate Analysis {#univariate-analysis}

Analyzing a single variable to understand its distribution and characteristics.

### 1.1 Histogram
**When to use:** Understand distribution of continuous numerical data
- Shows frequency distribution
- Best for: Age, Salary, Height, Weight
- Can reveal: Skewness, bimodality, gaps

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Simple histogram
axes[0].hist(df['Age'], bins=30, color='skyblue', edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Histogram: Age Distribution')
axes[0].axvline(df['Age'].mean(), color='red', linestyle='--', label=f'Mean: {df["Age"].mean():.1f}')
axes[0].axvline(df['Age'].median(), color='green', linestyle='--', label=f'Median: {df["Age"].median():.1f}')
axes[0].legend()

# Histogram with KDE
axes[1].hist(df['Salary'], bins=30, color='lightcoral', edgecolor='black', alpha=0.7, density=True)
df['Salary'].plot(kind='kde', ax=axes[1], color='darkred', linewidth=2)
axes[1].set_xlabel('Salary')
axes[1].set_ylabel('Density')
axes[1].set_title('Histogram with KDE: Salary Distribution')

plt.tight_layout()
plt.show()

print(f"Age - Skewness: {stats.skew(df['Age']):.2f}")
print(f"Age - Kurtosis: {stats.kurtosis(df['Age']):.2f}")

### 1.2 Kernel Density Estimation (KDE)
**When to use:** Smooth probability density estimation
- Shows smooth distribution curve
- Best for: Continuous data with moderate sample size
- Better than: Histogram for final reports

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

# Multiple KDE plots
df[df['Department'] == 'HR']['Salary'].plot(kind='kde', label='HR', linewidth=2)
df[df['Department'] == 'IT']['Salary'].plot(kind='kde', label='IT', linewidth=2)
df[df['Department'] == 'Finance']['Salary'].plot(kind='kde', label='Finance', linewidth=2)
df[df['Department'] == 'Sales']['Salary'].plot(kind='kde', label='Sales', linewidth=2)

ax.set_xlabel('Salary')
ax.set_ylabel('Density')
ax.set_title('KDE: Salary Distribution by Department')
ax.legend()
plt.show()

### 1.3 Box Plot
**When to use:** Identify outliers and understand quartiles
- Shows: Min, Q1, Median, Q3, Max
- Identifies: Outliers, skewness
- Best for: Quick statistical summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Single box plot
axes[0].boxplot(df['Age'], vert=True)
axes[0].set_ylabel('Age')
axes[0].set_title('Box Plot: Age Distribution')
axes[0].grid(True, alpha=0.3)

# Multiple box plots
df.boxplot(column='Salary', by='Department', ax=axes[1])
axes[1].set_xlabel('Department')
axes[1].set_ylabel('Salary')
axes[1].set_title('Box Plot: Salary by Department')
plt.suptitle('')  # Remove automatic title

plt.tight_layout()
plt.show()

### 1.4 Violin Plot
**When to use:** Show full distribution shape + outliers
- Combines: Box plot + KDE
- Better than: Box plot for non-normal distributions
- Shows: Multiple distributions comparison

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

sns.violinplot(data=df, x='Department', y='Salary', palette='Set2')
ax.set_xlabel('Department')
ax.set_ylabel('Salary')
ax.set_title('Violin Plot: Salary Distribution by Department')
plt.show()

print("Violin plots are excellent for comparing distributions across groups!")

### 1.5 Count Plot / Bar Chart
**When to use:** Count categorical data
- Shows: Frequency of categories
- Best for: Categorical variables
- Useful for: Understanding class distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Count plot
sns.countplot(data=df, x='Department', palette='husl', ax=axes[0])
axes[0].set_xlabel('Department')
axes[0].set_ylabel('Count')
axes[0].set_title('Count Plot: Employees by Department')
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height())}', 
                     (p.get_x() + p.get_width() / 2., p.get_height()),
                     ha='center', va='bottom')

# Performance distribution
df['Performance'].value_counts().sort_index().plot(kind='bar', ax=axes[1], color='steelblue')
axes[1].set_xlabel('Performance Rating')
axes[1].set_ylabel('Count')
axes[1].set_title('Bar Chart: Performance Ratings')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)

plt.tight_layout()
plt.show()

## 2. Bivariate Analysis {#bivariate-analysis}

Analyzing relationship between two variables.

### 2.1 Scatter Plot
**When to use:** Find relationship between two continuous variables
- Shows: Linear/non-linear relationships
- Identifies: Clusters, outliers
- Best for: Correlation exploration

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Simple scatter plot
axes[0].scatter(df['Experience'], df['Salary'], alpha=0.6, s=50, color='blue')
axes[0].set_xlabel('Experience (Years)')
axes[0].set_ylabel('Salary')
axes[0].set_title('Scatter Plot: Salary vs Experience')

# Add regression line
z = np.polyfit(df['Experience'], df['Salary'], 1)
p = np.poly1d(z)
axes[0].plot(df['Experience'].sort_values(), p(df['Experience'].sort_values()), 
             "r--", linewidth=2, label='Trend Line')
axes[0].legend()

# Scatter plot with hue
for dept in df['Department'].unique():
    mask = df['Department'] == dept
    axes[1].scatter(df[mask]['Age'], df[mask]['Salary'], label=dept, alpha=0.6, s=50)

axes[1].set_xlabel('Age')
axes[1].set_ylabel('Salary')
axes[1].set_title('Scatter Plot: Salary vs Age by Department')
axes[1].legend()

plt.tight_layout()
plt.show()

# Calculate correlation
corr = df['Experience'].corr(df['Salary'])
print(f"Correlation between Experience and Salary: {corr:.3f}")

### 2.2 Line Plot
**When to use:** Show trends over time or continuous variable
- Shows: Trends, patterns
- Best for: Time series data
- Multiple lines: Compare multiple series

In [ ]:
# Aggregate by date
daily_avg = df.groupby('Date')['Salary'].mean()
daily_count = df.groupby('Date').size()

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Line plot
daily_avg.plot(ax=axes[0], color='green', linewidth=2)
axes[0].fill_between(daily_avg.index, daily_avg, alpha=0.3, color='green')
axes[0].set_ylabel('Average Salary')
axes[0].set_title('Line Plot: Average Salary Over Time')
axes[0].grid(True, alpha=0.3)

# Multiple lines
daily_count.plot(ax=axes[1], color='orange', linewidth=2, label='Records')
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Count')
axes[1].set_title('Line Plot: Records Over Time')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 2.3 Hexbin Plot
**When to use:** Large datasets with overlapping points
- Shows: Density of overlapping points
- Best for: Big data visualization
- Alternative to: Scatter plot with many points

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

hexbin = ax.hexbin(df['Age'], df['Salary'], gridsize=15, cmap='YlOrRd', mincnt=1)
ax.set_xlabel('Age')
ax.set_ylabel('Salary')
ax.set_title('Hexbin Plot: Salary vs Age (Density)')
cb = plt.colorbar(hexbin, ax=ax)
cb.set_label('Count')
plt.show()

## 3. Multivariate Analysis {#multivariate-analysis}

Analyzing relationships between 3+ variables simultaneously.

### 3.1 Heatmap / Correlation Matrix
**When to use:** Visualize correlations between multiple variables
- Shows: Pairwise correlations
- Color intensity: Strength of correlation
- Best for: Selecting features, finding multicollinearity

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Correlation matrix
numeric_cols = df.select_dtypes(include=[np.number]).columns
corr_matrix = df[numeric_cols].corr()

# Heatmap
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, ax=axes[0], cbar_kws={'label': 'Correlation'})
axes[0].set_title('Heatmap: Correlation Matrix')

# Masked heatmap (upper triangle only)
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, ax=axes[1], cbar_kws={'label': 'Correlation'})
axes[1].set_title('Heatmap: Lower Triangle Only')

plt.tight_layout()
plt.show()

print("\nCorrelation Matrix:")
print(corr_matrix)

### 3.2 Pairplot
**When to use:** Explore relationships between all pairs of variables
- Shows: Univariate and bivariate distributions
- Best for: Small number of variables (3-5)
- Reveals: Patterns, clusters, outliers

In [ ]:
# Create a smaller dataset for pairplot
df_pair = df[['Age', 'Experience', 'Salary', 'Performance']].head(100)

pairplot = sns.pairplot(df_pair, diag_kind='kde', plot_kws={'alpha': 0.6, 's': 40})
pairplot.fig.suptitle('Pairplot: All Variables Relationships', y=1.001)
plt.tight_layout()
plt.show()

### 3.3 3D Scatter Plot
**When to use:** Visualize 3 continuous variables simultaneously
- Shows: 3D relationships
- Best for: Medium-sized datasets
- Can be: Hard to interpret, use with caution

In [ ]:
from mpl_toolkits.mplot3d import Axes3D

fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')

# Color by department
departments = df['Department'].unique()
colors = ['red', 'blue', 'green', 'orange']

for dept, color in zip(departments, colors):
    mask = df['Department'] == dept
    ax.scatter(df[mask]['Age'], df[mask]['Experience'], df[mask]['Salary'],
              c=color, label=dept, alpha=0.6, s=30)

ax.set_xlabel('Age')
ax.set_ylabel('Experience')
ax.set_zlabel('Salary')
ax.set_title('3D Scatter Plot: Age, Experience, and Salary')
ax.legend()
plt.show()

## 4. Distribution Analysis {#distribution-analysis}

### 4.1 Q-Q Plot
**When to use:** Check if data follows normal distribution
- Shows: Points should align with diagonal line for normal distribution
- Best for: Validating normality assumption
- Deviation: Indicates non-normality

In [ ]:
from scipy.stats import probplot

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Q-Q plot for Age
probplot(df['Age'], dist="norm", plot=axes[0])
axes[0].set_title('Q-Q Plot: Age')

# Q-Q plot for Salary
probplot(df['Salary'], dist="norm", plot=axes[1])
axes[1].set_title('Q-Q Plot: Salary')

plt.tight_layout()
plt.show()

### 4.2 Distribution Comparison
**When to use:** Compare distribution across groups

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Histogram + KDE
axes[0, 0].hist(df['Age'], bins=30, density=True, alpha=0.7, color='skyblue')
df['Age'].plot(kind='kde', ax=axes[0, 0], color='blue', linewidth=2)
axes[0, 0].set_title('Age Distribution')

# Box plot
axes[0, 1].boxplot(df['Salary'])
axes[0, 1].set_title('Salary Distribution')

# Distribution by group
for dept in df['Department'].unique():
    data = df[df['Department'] == dept]['Salary']
    axes[1, 0].hist(data, alpha=0.5, label=dept, bins=15)
axes[1, 0].set_title('Salary Distribution by Department')
axes[1, 0].legend()
axes[1, 0].set_xlabel('Salary')

# Strip plot
sns.stripplot(data=df, x='Department', y='Salary', ax=axes[1, 1], jitter=True, size=6)
axes[1, 1].set_title('Salary by Department (Strip Plot)')

plt.tight_layout()
plt.show()

## 5. Correlation Analysis {#correlation-analysis}

In [ ]:
# Calculate various correlations
pearson_corr = df['Age'].corr(df['Salary'])
spearman_corr = df['Age'].corr(df['Salary'], method='spearman')

print(f"Pearson Correlation (Age vs Salary): {pearson_corr:.3f}")
print(f"Spearman Correlation (Age vs Salary): {spearman_corr:.3f}")

# Full correlation matrix
print("\nFull Correlation Matrix:")
print(df[numeric_cols].corr())

### Joint Plot
**When to use:** Show bivariate distribution with marginals
- Shows: Scatter plot + univariate distributions
- Best for: Understanding 2D relationships

In [ ]:
fig = sns.jointplot(data=df, x='Experience', y='Salary', kind='hex', height=8, 
                   marginal_kws=dict(bins=30, fill=True))
fig.fig.suptitle('Joint Plot: Experience vs Salary (with Marginals)', y=1.00)
plt.show()

# Alternative: regression joint plot
fig = sns.jointplot(data=df, x='Age', y='Salary', kind='reg', height=8, 
                   marginal_kws=dict(bins=30, fill=True))
fig.fig.suptitle('Joint Plot: Age vs Salary (with Regression)', y=1.00)
plt.show()

## 6. Categorical Data Analysis {#categorical-data-analysis}

### 6.1 Categorical Relationships

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Count plot
sns.countplot(data=df, x='Department', palette='Set1', ax=axes[0, 0])
axes[0, 0].set_title('Count Plot: Department Distribution')

# Grouped count plot
df['Performance_Category'] = pd.cut(df['Performance'], bins=[0, 2, 4, 5], 
                                     labels=['Low', 'Medium', 'High'])
sns.countplot(data=df, x='Department', hue='Performance_Category', ax=axes[0, 1])
axes[0, 1].set_title('Count Plot: Department by Performance')

# Cross-tabulation heatmap
crosstab = pd.crosstab(df['Department'], df['Performance_Category'])
sns.heatmap(crosstab, annot=True, fmt='d', cmap='Blues', ax=axes[1, 0])
axes[1, 0].set_title('Heatmap: Department vs Performance')

# Normalized heatmap
crosstab_norm = crosstab.div(crosstab.sum(axis=1), axis=0)
sns.heatmap(crosstab_norm, annot=True, fmt='.2%', cmap='RdYlGn', ax=axes[1, 1])
axes[1, 1].set_title('Heatmap: Normalized Distribution')

plt.tight_layout()
plt.show()

## 7. Time Series Analysis {#time-series-analysis}

In [ ]:
# Create time series data
ts_data = df.groupby('Date').agg({
    'Salary': 'mean',
    'Age': 'mean',
    'Experience': 'mean'
})

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Time series plot
ts_data['Salary'].plot(ax=axes[0], linewidth=2, color='blue')
axes[0].fill_between(ts_data.index, ts_data['Salary'], alpha=0.3, color='blue')
axes[0].set_ylabel('Average Salary')
axes[0].set_title('Time Series: Average Salary Over Time')
axes[0].grid(True, alpha=0.3)

# Multiple time series
ts_data.plot(ax=axes[1], linewidth=2)
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Average Value')
axes[1].set_title('Time Series: Multiple Variables')
axes[1].grid(True, alpha=0.3)
axes[1].legend(loc='best')

plt.tight_layout()
plt.show()

## 8. Outlier Detection {#outlier-detection}

In [ ]:
from scipy.stats import zscore

# Identify outliers using Z-score
z_scores = np.abs(zscore(df['Salary']))
outliers = df[z_scores > 3]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Box plot for outlier visualization
box_plot = axes[0].boxplot(df['Salary'], vert=True)
axes[0].scatter([1]*len(outliers), outliers['Salary'], color='red', s=100, 
               label='Outliers', zorder=3)
axes[0].set_ylabel('Salary')
axes[0].set_title('Box Plot: Outlier Detection')
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis='y')

# Scatter plot with Z-score
colors = ['red' if z > 3 else 'blue' for z in z_scores]
axes[1].scatter(range(len(df)), z_scores, c=colors, alpha=0.6, s=30)
axes[1].axhline(y=3, color='red', linestyle='--', label='Z-score threshold')
axes[1].axhline(y=-3, color='red', linestyle='--')
axes[1].set_xlabel('Record Index')
axes[1].set_ylabel('Z-Score')
axes[1].set_title('Z-Score: Outlier Detection')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Number of outliers (|Z| > 3): {len(outliers)}")

## Summary: Visualization Selection Guide

| Situation | Best Visualization | Alternative |
|-----------|-------------------|-------------|
| **Single numerical variable** | Histogram | KDE, Box Plot |
| **Distribution shape** | KDE or Violin Plot | Histogram |
| **Detect outliers** | Box Plot | Scatter + Z-score |
| **Two continuous variables** | Scatter Plot | Hexbin, Joint Plot |
| **Categorical count** | Count Plot / Bar Chart | Pie Chart |
| **Correlation matrix** | Heatmap | Pairplot |
| **Multiple relationships** | Pairplot | Facet Grid |
| **Time series** | Line Plot | Area Chart |
| **Distribution by group** | Violin Plot | Box Plot, Histogram |
| **Categorical relationships** | Crosstab Heatmap | Grouped Bar |
| **2D distribution + margins** | Joint Plot | 2D Histogram |
| **3 variables** | 3D Scatter | Heatmap + grouping |
| **Density with overlaps** | Hexbin | Scatter with alpha |

### Key Principles:
1. **Know your data type** - Numerical vs Categorical
2. **Choose clarity over complexity** - Simple plots beat complicated ones
3. **Consider audience** - Technical vs Non-technical
4. **Use color purposefully** - For grouping or intensity, not decoration
5. **Avoid chart junk** - Remove unnecessary elements
6. **Label everything** - Axes, title, legend
7. **Test interpretability** - Can someone understand it quickly?